In [ ]:
!pip install tensorflow=='2.15' --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorstore 0.1.67 requires ml-dtypes>=0.3.1, but you have ml-dtypes 0.2.0 which is incompatible.
tf-keras 2.17.0 requires tensorflow<2.18,>=2.17, but you have tensorflow 2.15.0 which is incompatible.


## Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import (Conv2D, BatchNormalization, ReLU, Add,
                                     GlobalAveragePooling2D, Dense, Input,
                                     Concatenate, Layer, Maximum, Dropout, Multiply)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import ImageFile

## Model Parameters

In [ ]:
IMG_HEIGHT = 256
IMG_WIDTH = 256
CHANNELS = 3
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)

BATCH_SIZE = 256
N_EPOCHS = 10
LR = 1e-3

VALUE_DT = 0.2
PATH_DIR = ''

ImageFile.LOAD_TRUNCATED_IMAGES = True

## Model Functions

In [ ]:
class RGBtoHSV(Layer):
    def __init__(self, **kwargs):
        super(RGBtoHSV, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.image.rgb_to_hsv(inputs)

    def get_config(self):
        config = super(RGBtoHSV, self).get_config()
        return config

class RGBtoYCbCr(Layer):
    def __init__(self, **kwargs):
        super(RGBtoYCbCr, self).__init__(**kwargs)

    def call(self, inputs):

        rgb_to_ycbcr_kernel = tf.constant([[0.299, 0.587, 0.114],
                                           [-0.1687, -0.3313, 0.5],
                                           [0.5, -0.4187, -0.0813]])
        offset = tf.constant([0, 128/255, 128/255], dtype=tf.float32)
        ycbcr = tf.tensordot(inputs, rgb_to_ycbcr_kernel, axes=[[3], [1]]) + offset

        return ycbcr

    def get_config(self):
        config = super(RGBtoYCbCr, self).get_config()
        return config

In [ ]:
class FuzzyPooling(Layer):
    def __init__(self, pool_size=(2, 2), strides=None, padding='VALID', fuzzy_k=2, **kwargs):
        super(FuzzyPooling, self).__init__(**kwargs)
        self.pool_size = pool_size
        self.strides = strides if strides is not None else pool_size
        self.padding = padding.upper()
        self.fuzzy_k = fuzzy_k

    def call(self, inputs):

        patches = tf.image.extract_patches(
            images=inputs,
            sizes=[1, self.pool_size[0], self.pool_size[1], 1],
            strides=[1, self.strides[0], self.strides[1], 1],
            rates=[1, 1, 1, 1],
            padding=self.padding
        )

        batch_size = tf.shape(inputs)[0]
        new_height = tf.shape(patches)[1]
        new_width = tf.shape(patches)[2]
        channels = inputs.shape[-1]
        patch_dim = self.pool_size[0] * self.pool_size[1]

        patches = tf.reshape(patches, [batch_size, new_height, new_width, channels, patch_dim])


        max_vals = tf.reduce_max(patches, axis=-1, keepdims=True)
        min_vals = tf.reduce_min(patches, axis=-1, keepdims=True)
        denom = max_vals - min_vals + 1e-6

        membership = 1 - tf.abs(patches - max_vals) / denom
        membership = tf.pow(membership, self.fuzzy_k)

        numerator = tf.reduce_sum(patches * membership, axis=-1)
        denominator = tf.reduce_sum(membership, axis=-1) + 1e-6
        fuzzy_pool = numerator / denominator

        return fuzzy_pool

    def compute_output_shape(self, input_shape):

        if self.padding == 'VALID':
            out_height = (input_shape[1] - self.pool_size[0]) // self.strides[0] + 1
            out_width = (input_shape[2] - self.pool_size[1]) // self.strides[1] + 1
        elif self.padding == 'SAME':
            out_height = (input_shape[1] + self.strides[0] - 1) // self.strides[0]
            out_width = (input_shape[2] + self.strides[1] - 1) // self.strides[1]
        else:
            raise ValueError(f"Invalid padding type: {self.padding}")

        return (input_shape[0], out_height, out_width, input_shape[3])

    def get_config(self):

        config = super(FuzzyPooling, self).get_config()
        config.update({
            'pool_size': self.pool_size,
            'strides': self.strides,
            'padding': self.padding,
            'fuzzy_k': self.fuzzy_k,
        })
        return config

In [ ]:
class GlobalFuzzyPooling2D(Layer):
    def __init__(self, fuzzy_k=2, **kwargs):
        super(GlobalFuzzyPooling2D, self).__init__(**kwargs)
        self.fuzzy_k = fuzzy_k

    def call(self, inputs):

        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        channels = inputs.shape[3]

        inputs_flat = tf.reshape(inputs, [batch_size, height * width, channels])

        max_vals = tf.reduce_max(inputs_flat, axis=1, keepdims=True)
        min_vals = tf.reduce_min(inputs_flat, axis=1, keepdims=True)
        denom = max_vals - min_vals + 1e-6

        membership = 1 - tf.abs(inputs_flat - max_vals) / denom
        membership = tf.pow(membership, self.fuzzy_k)

        numerator = tf.reduce_sum(inputs_flat * membership, axis=1)
        denominator = tf.reduce_sum(membership, axis=1) + 1e-6
        fuzzy_global_pool = numerator / denominator

        return fuzzy_global_pool

    def compute_output_shape(self, input_shape):

        return (input_shape[0], input_shape[3])

    def get_config(self):

        config = super(GlobalFuzzyPooling2D, self).get_config()
        config.update({
            'fuzzy_k': self.fuzzy_k,
        })
        return config

In [ ]:
def residual_block(x, filters, stride=1):

    shortcut = x

    x = Conv2D(filters, kernel_size=(3, 3), strides=stride, padding="same")(x)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, kernel_size=(3, 3), strides=1, padding="same")(x)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, kernel_size=(1, 1), strides=stride, padding="same")(shortcut)
        shortcut = Dropout(VALUE_DT)(shortcut, training=True)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x, shortcut])
    x = ReLU()(x)

    return x

In [ ]:
def backbone_resnet18(x):
    x = Conv2D(64, kernel_size=(7, 7), strides=2, padding="same")(x)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = FuzzyPooling(pool_size=(3, 3), strides=(2, 2), padding='SAME')(x)

    x = residual_block(x, 64, stride=1)
    x = residual_block(x, 64, stride=1)

    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128, stride=1)

    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256, stride=1)

    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512, stride=1)

    return x

In [ ]:
def phd_resnet18(input_shape=INPUT_SHAPE):
    inputs = Input(shape=input_shape)

    hsv_image = RGBtoHSV()(inputs)

    ycbcr_image = RGBtoYCbCr()(inputs)

    concatenated_inputs = Concatenate()([inputs, hsv_image, ycbcr_image])

    paper_branch = backbone_resnet18(concatenated_inputs)
    replay_branch = backbone_resnet18(concatenated_inputs)
    mask_branch = backbone_resnet18(concatenated_inputs)

    liveness_branch = backbone_resnet18(concatenated_inputs)


    x = GlobalFuzzyPooling2D()(paper_branch)

    paper_output = Dense(2, activation='softmax', name='paper_output')(x)

    x = GlobalFuzzyPooling2D()(replay_branch)
    replay_output = Dense(2, activation='softmax', name='replay_output')(x)

    x = GlobalFuzzyPooling2D()(mask_branch)
    mask_output = Dense(2, activation='softmax', name='mask_output')(x)

    x = GlobalFuzzyPooling2D()(liveness_branch)
    liveness_output = Dense(2, activation='softmax', name='liveness_output')(x)

    concatenated_spoofs = Multiply()([paper_branch, replay_branch, mask_branch])

    x = Conv2D(512, kernel_size=(3, 3), padding="same")(concatenated_spoofs)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    concatenated_liveness = Concatenate()([x, liveness_branch])

    x = Conv2D(512, kernel_size=(3, 3), padding="same")(concatenated_liveness)
    x = Dropout(VALUE_DT)(x, training=True)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = GlobalFuzzyPooling2D()(x)
    liveness_final_output = Dense(2, activation='softmax', name='liveness_final_output')(x)

    model = tf.keras.models.Model(inputs, [paper_output, mask_output, replay_output, liveness_output, liveness_final_output])
    return model


#### Load model

In [ ]:
!cp "/content/gdrive/MyDrive/PhD_Models/saved_model/ResNet-18-MCD-FP_Final.keras" .

In [ ]:
model = tf.keras.models.load_model('ResNet-18-MCD-FP_Final.keras', custom_objects={
                                        'RGBtoHSV': RGBtoHSV,
                                       'RGBtoYCbCr': RGBtoYCbCr,
                                       'FuzzyPooling': FuzzyPooling,
                                       'GlobalFuzzyPooling2D': GlobalFuzzyPooling2D,
})
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 rg_bto_hsv (RGBtoHSV)       (None, 256, 256, 3)          0         ['input_1[0][0]']             
                                                                                                  
 rg_bto_y_cb_cr (RGBtoYCbCr  (None, 256, 256, 3)          0         ['input_1[0][0]']             
 )                                                                                                
                                                                                                  
 concatenate (Concatenate)   (None, 256, 256, 9)          0         ['input_1[0][0]',         

## Load Data

In [ ]:
import glob

In [ ]:
!cp "/content/gdrive/MyDrive/PhD/datasets/iqa_spoof_dataset/fas_iqa_dataset.zip" .
!unzip -qq fas_iqa_dataset.zip

In [ ]:
np.random.seed(42)
df = pd.DataFrame([
    {
        'filename': x.split('/')[-1],
        'scene': x.split('/')[-2],
        'label': x.split('/')[-3],
        'data_type': x.split('/')[-4],
        'distortion_type': x.split('/')[-5],
        'label_iqa': x.split('/')[-1].split('_f_')[0] if len(x.split('/')[-1].split('_f_'))>1 else 'original',
        'full_path_iqa': x,
    } for x in glob.glob('fas_iqa_dataset/*/*/*/*/*')
])
print('Size DF:', df.shape[0])
df.sample(3)

Size DF: 100000


,filename,scene,label,data_type,distortion_type,label_iqa,full_path_iqa
75721,BlurY_7_f_270542.jpg,client002268_Env1_Ilum1_Spt3,attack,train,BlurY,BlurY_7,fas_iqa_dataset/BlurY/train/attack/client00226...
80184,hbright_10_f_frame_182.jpg,real_client012_laptop_SD_scene01,real,train,hbright,hbright_10,fas_iqa_dataset/hbright/train/real/real_client...
19864,noise_25_f_frame_153.jpg,attack_client008_android_SD_ipad_video_scene01,attack,train,noise,noise_25,fas_iqa_dataset/noise/train/attack/attack_clie...


## Prepare Data

In [ ]:
val_datagen = ImageDataGenerator(rescale=1./255)

# test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
val_generator = val_datagen.flow_from_dataframe(
    df,
    '',
    x_col='full_path_iqa',
    y_col='label_iqa',
    target_size=IMG_SIZE,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 100000 validated image filenames belonging to 25 classes.


In [ ]:
import tqdm
np.random.seed(42)
tf.random.set_seed(42)
no_times=25
for i in tqdm.tqdm(range(no_times), total=no_times):
    predict = np.array(model.predict(val_generator, verbose=1))[:,:,1:]

    if i==0:
        predict_test_array = predict.copy()
    else:
        predict_test_array = np.concatenate([predict_test_array, predict], axis=-1)

  0%|          | 0/25 [00:00<?, ?it/s]

391/391 [==============================] - 246s 602ms/step


  4%|▍         | 1/25 [04:07<1:38:59, 247.48s/it]

391/391 [==============================] - 233s 596ms/step


  8%|▊         | 2/25 [08:01<1:31:53, 239.70s/it]

...

 92%|█████████▏| 23/25 [1:30:02<07:48, 234.33s/it]

391/391 [==============================] - 233s 596ms/step


 96%|█████████▌| 24/25 [1:33:57<03:54, 234.40s/it]

391/391 [==============================] - 233s 596ms/step


100%|██████████| 25/25 [1:37:51<00:00, 234.87s/it]


In [ ]:
df['paper_pred'] = predict_test_array[0].tolist()
df['mask_pred'] = predict_test_array[1].tolist()
df['replay_pred'] = predict_test_array[2].tolist()
df['liveness_pred'] = predict_test_array[3].tolist()
df['liveness_final_pred'] = predict_test_array[4].tolist()
df.to_csv(f'/content/gdrive/MyDrive/csv_results/PhD4_protIQA_trCeAS_tsIQA.csv', index=False)
df.head(3)

,filename,scene,label,data_type,distortion_type,label_iqa,full_path_iqa,paper_pred,mask_pred,replay_pred,liveness_pred,liveness_final_pred
0,jpgcompression_30_f_523234.jpg,client005536_Env0_Ilum0_Spt0,real,test,jpgcompression,jpgcompression_30,fas_iqa_dataset/jpgcompression/test/real/clien...,"[0.9977341890335083, 0.9969382286071777, 0.997...","[0.9866105914115906, 0.9874058365821838, 0.981...","[0.9994205236434937, 0.9993859529495239, 0.999...","[0.9998289346694946, 0.9999500513076782, 0.999...","[0.9998968839645386, 0.9999494552612305, 0.999..."
1,jpgcompression_10_f_523234.jpg,client005536_Env0_Ilum0_Spt0,real,test,jpgcompression,jpgcompression_10,fas_iqa_dataset/jpgcompression/test/real/clien...,"[0.9999243021011353, 0.9999902248382568, 0.999...","[0.9945197701454163, 0.9950041174888611, 0.995...","[0.9981977343559265, 0.9985370635986328, 0.997...","[0.9999407529830933, 0.9999293088912964, 0.999...","[0.9999637603759766, 0.9999723434448242, 0.999..."
2,jpgcompression_50_f_523234.jpg,client005536_Env0_Ilum0_Spt0,real,test,jpgcompression,jpgcompression_50,fas_iqa_dataset/jpgcompression/test/real/clien...,"[0.991926372051239, 0.9739698767662048, 0.8457...","[0.9673277139663696, 0.826004147529602, 0.9621...","[0.9929105639457703, 0.9986376166343689, 0.996...","[0.9994391798973083, 0.9991500377655029, 0.999...","[0.9987591505050659, 0.9996721744537354, 0.999..."


END